# Softmax, entropía cruzada y pérdida probabilística

**Capítulo 2 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_linear-classification/softmax-regression.ipynb` · [Lección original](https://d2l.ai/chapter_linear-classification/softmax-regression.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Regresión de Softmax
<a id="sec_softmax"></a>

En [Referencia sec_linear_regression](https://d2l.ai/chapter_linear-regression/linear-regression.html#sec-linear-regression), introdujimos regresión lineal, trabajando a través de implementaciones desde cero en [Referencia sec_linear_scratch](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html#sec-linear-scratch) y de nuevo utilizando API de alto nivel de un biblioteca de aprendizaje profundo en [Referencia sec_linear_concise](https://d2l.ai/chapter_linear-regression/linear-regression-concise.html#sec-linear-concise) para hacer el levantamiento pesado.

La regresión es el martillo al que llegamos cuando queremos responder a *¿cuánto?* o *¿cuántas preguntas?*. Si quieres predecir el número de dólares (precio) en el que se venderá una casa, o el número de victorias que un equipo de béisbol podría tener, o el número de días que un paciente permanecerá hospitalizado antes de ser dado de alta, entonces probablemente estás buscando un modelo de regresión. Sin embargo, incluso dentro de los modelos de regresión, hay distinciones importantes. Por ejemplo, el precio de una casa nunca será negativo y los cambios a menudo podrían ser *relativos* a su precio de referencia. Como tal, podría ser más eficaz retroceder en el logaritmo del precio. Del mismo modo, el número de días que un paciente pasa en el hospital es una variable aleatoria no negativa *discreta. Como tal, los cuadrados mínimos podrían no ser un enfoque ideal. Este tipo de modelado del tiempo al evento viene con una serie de otras complicaciones que se tratan en un subcampo especializado llamado *modelo de supervivencia*.

El punto aquí no es abrumarle, pero sólo para hacerle saber que hay mucho más que estimar que simplemente minimizar los errores al cuadrado. Y más ampliamente, hay mucho más que aprendizaje supervisado que regresión. En esta sección, nos centramos en *clasificación* problemas donde ponemos a un lado * ¿cuánto? * preguntas y en lugar de centrarse en * qué categoría? * preguntas.

* ¿Este correo electrónico pertenece a la carpeta de spam o a la bandeja de entrada?
* ¿Es este cliente más propenso a registrarse
o no inscribirse en un servicio de suscripción?
* ¿ Representa esta imagen un burro, un perro, un gato o un gallo?
* ¿Qué película es más probable que Aston vea la próxima?
* ¿Qué sección del libro vas a leer a continuación?

Coloquialmente, los practicantes de aprendizaje automático sobrecargan la palabra *clasificación* para describir dos problemas sutilmente diferentes: (i) aquellos en los que estamos interesados sólo en asignaciones difíciles de ejemplos a categorías (clases); y (ii) aquellos en los que deseamos hacer asignaciones suaves, es decir, para evaluar la probabilidad de que se aplique cada categoría. La distinción tiende a ser borrosa, en parte, porque a menudo, incluso cuando sólo nos preocupan las asignaciones duras, seguimos utilizando modelos que hacen asignaciones suaves.

Aún más, hay casos en los que más de una etiqueta podría ser cierta. Por ejemplo, un artículo de noticias podría cubrir simultáneamente los temas de entretenimiento, negocios y vuelos espaciales, pero no los temas de medicina o deportes. Por lo tanto, categorizar en una de las categorías anteriores por su cuenta no sería muy útil. Este problema se conoce comúnmente como [multi-label classification](https://en.wikipedia.org/wiki/Multi-label_classification). Vea [Tsoumakas.Katakis.2007](https://d2l.ai/chapter_references/zreferences.html) para una visión general y [Huang.Xu.Yu.2015](https://d2l.ai/chapter_references/zreferences.html) para un algoritmo eficaz al etiquetar imágenes.

## Clasificación
<a id="subsec_classification-problem"></a>

Para mojarnos los pies, empecemos con un simple problema de clasificación de imágenes. Aquí, cada entrada consiste en una imagen de escala de grises $2\times2$. Podemos representar cada valor de píxel con un solo escalar, dándonos cuatro características $x_1, x_2, x_3, x_4$. Además, supongamos que cada imagen pertenece a una de las categorías "gato", "pollo" y "perro".

A continuación, tenemos que elegir cómo representar las etiquetas. Tenemos dos opciones obvias. Tal vez el impulso más natural sería elegir $y \in \{1, 2, 3\}$, donde los números enteros representan $\{\textrm{dog}, \textrm{cat}, \textrm{chicken}\}$ respectivamente. Esta es una gran manera de * storing* tal información en un ordenador. Si las categorías tenían algún orden natural entre ellos, digamos si estábamos tratando de predecir $\{\textrm{baby}, \textrm{toddler}, \textrm{adolescent}, \textrm{young adult}, \textrm{adult}, \textrm{geriatric}\}$, entonces incluso podría tener sentido lanzar esto como un problema [ordinal regression](https://en.wikipedia.org/wiki/Ordinal_regression) y mantener las etiquetas en este formato. Vea [Moon.Smola.Chang.ea.2010](https://d2l.ai/chapter_references/zreferences.html) para una visión general de diferentes tipos de funciones de pérdida de clasificación y [Beutel.Murray.Faloutsos.ea.2014](https://d2l.ai/chapter_references/zreferences.html) para un enfoque bayesiano que aborda las respuestas con más de un modo.

En general, los problemas de clasificación no vienen con los pedidos naturales entre las clases. Afortunadamente, los estadísticos hace mucho tiempo inventaron una forma sencilla de representar datos categóricas: la *codificación de un solo calor*. Una codificación de un solo calor es un vector con tantos componentes como tenemos categorías. El componente correspondiente a la categoría de una instancia en particular se establece en 1 y todos los demás componentes se establecen en 0. En nuestro caso, una etiqueta $y$ sería un vector tridimensional, con $(1, 0, 0)$ correspondiente a "cat", $(0, 1, 0)$ a "chicken", y $(0, 0, 1)$ a "dog":

$$y \in \{(1, 0, 0), (0, 1, 0), (0, 0, 1)\}.$$

### Modelo lineal
Para estimar las probabilidades condicionales asociadas con todas las clases posibles, necesitamos un modelo con múltiples salidas, una por clase. Para abordar la clasificación con modelos lineales, necesitaremos tantas funciones afín como tengamos salidas. Estrictamente hablando, sólo necesitamos una menos, ya que la categoría final tiene que ser la diferencia entre $1$ y la suma de las otras categorías, pero por razones de simetría utilizamos una parametrización ligeramente redundante. Cada salida corresponde a su propia función afín. En nuestro caso, ya que tenemos 4 características y 3 posibles categorías de salida, necesitamos 12 escalares para representar los pesos ($w$ con subíndices), y 3 escalares para representar los sesgos ($b$ con subíndices).

$$
\begin{aligned}
o_1 &= x_1 w_{11} + x_2 w_{12} + x_3 w_{13} + x_4 w_{14} + b_1,\\
o_2 &= x_1 w_{21} + x_2 w_{22} + x_3 w_{23} + x_4 w_{24} + b_2,\\
o_3 &= x_1 w_{31} + x_2 w_{32} + x_3 w_{33} + x_4 w_{34} + b_3.
\end{aligned}
$$

El diagrama de red neural correspondiente se muestra en [Referencia fig_softmaxreg](https://d2l.ai/chapter_linear-classification/softmax-regression.html#fig-softmaxreg). Al igual que en la regresión lineal, utilizamos una red neural de una sola capa. Y puesto que el cálculo de cada salida, $o_1, o_2$ y $o_3$, depende de cada entrada, $x_1$, $x_2$, $x_3$ y $x_4$, la capa de salida también puede describirse como una *capa totalmente conectada*.

![La regresión softmax es una red neuronal de una sola capa.](../recursos/originales/softmaxreg.svg)
<a id="fig_softmaxreg"></a>

Para una notación más concisa utilizamos vectores y matrices: $\mathbf{o} = \mathbf{W} \mathbf{x} + \mathbf{b}$ es mucho más adecuado para las matemáticas y el código. Tenga en cuenta que hemos reunido todos nuestros pesos en una matriz $3 \times 4$ y todos los sesgos $\mathbf{b} \in \mathbb{R}^3$ en un vector.

### El Softmax
<a id="subsec_softmax_operation"></a>

Asumiendo una función de pérdida adecuada, podríamos intentar, directamente, minimizar la diferencia entre $\mathbf{o}$ y las etiquetas $\mathbf{y}$. Si bien resulta que tratar la clasificación como un problema de regresión valorada por vectores funciona sorprendentemente bien, no obstante es insatisfactorio de las siguientes maneras:

* No hay garantía de que las salidas $o_i$ suman hasta $1$ en la forma en que esperamos que se comporten.
* No hay garantía de que las salidas $o_i$ sean incluso no negativas, incluso si sus salidas suman hasta $1$, o que no superen $1$.

Ambos aspectos hacen el problema de estimación difícil de resolver y la solución muy frágil a los valores atípicos. Por ejemplo, si suponemos que hay una dependencia lineal positiva entre el número de dormitorios y la probabilidad de que alguien va a comprar una casa, la probabilidad podría superar $1$ cuando se trata de comprar una mansión! Como tal, necesitamos un mecanismo para "aplastar" las salidas.

Por ejemplo, podríamos asumir que las salidas $\mathbf{o}$ son versiones corruptas de $\mathbf{y}$, donde la corrupción se produce mediante la adición de ruido $\boldsymbol{\epsilon}$ extraído de una distribución normal. En otras palabras, $\mathbf{y} = \mathbf{o} + \boldsymbol{\epsilon}$, donde $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$. Este es el llamado [probit model](https://en.wikipedia.org/wiki/Probit_model), introducido por primera vez por [Fechner.1860](https://d2l.ai/chapter_references/zreferences.html). Aunque atractivo, no funciona bien ni conduce a un problema de optimización particularmente agradable, en comparación con el softmax.

Otra manera de lograr este objetivo (y de asegurar la no negatividad) es utilizar una función exponencial $P(y = i) \propto \exp o_i$. Esto satisface el requisito de que la probabilidad de clase condicional aumenta con el aumento de $o_i$, es monotónico, y todas las probabilidades no son negativas. Podemos entonces transformar estos valores para que sumen hasta $1$ dividiendo cada uno por su suma. Este proceso se llama *normalización*. Poner estas dos piezas juntas nos da la función *softmax*:

$$\hat{\mathbf{y}} = \mathrm{softmax}(\mathbf{o}) \quad \textrm{where}\quad \hat{y}_i = \frac{\exp(o_i)}{\sum_j \exp(o_j)}.$$

:eqlabel:`eq_softmax_y_and_o`

Tenga en cuenta que la coordenada más grande de $\mathbf{o}$ corresponde a la clase más probable según $\hat{\mathbf{y}}$. Además, debido a que la operación softmax conserva el orden entre sus argumentos, no necesitamos calcular el softmax para determinar qué clase se le ha asignado la mayor probabilidad.

$$
\operatorname*{argmax}_j \hat y_j = \operatorname*{argmax}_j o_j.
$$

La idea de un softmax se remonta a [Gibbs.1902](https://d2l.ai/chapter_references/zreferences.html), que adaptó ideas de la física. Fechado aún más atrás, Boltzmann, el padre de la física estadística moderna, utilizó este truco para modelar una distribución sobre estados energéticos en moléculas de gas. En particular, descubrió que la prevalencia de un estado de energía en un conjunto termodinámico, como las moléculas en un gas, es proporcional a $\exp(-E/kT)$. Aquí, $E$ es la energía de un estado, $T$ es la temperatura, y $k$ es la constante Boltzmann. Cuando los estadísticos hablan de aumentar o disminuir la "temperatura" de un sistema estadístico, se refieren a cambiar $T$ para favorecer estados energéticos más bajos o más altos. Siguiendo la idea de Gibbs, la energía equivale a error. Los modelos [Ranzato.Boureau.Chopra.ea.2007](https://d2l.ai/chapter_references/zreferences.html) basados en energía utilizan este punto de vista al describir problemas en el aprendizaje profundo.

### Vectorización
<a id="subsec_softmax_vectorization"></a>

Para mejorar la eficiencia computacional, vectorizamos cálculos en minibatches de datos. Supongamos que se nos da un minibatch $\mathbf{X} \in \mathbb{R}^{n \times d}$ de ejemplos $n$ con dimensionalidad (número de entradas) $d$. Además, supongamos que tenemos categorías $q$ en la salida. Entonces los pesos satisfacen $\mathbf{W} \in \mathbb{R}^{d \times q}$ y el sesgo satisface $\mathbf{b} \in \mathbb{R}^{1\times q}$.

$$ \begin{aligned} \mathbf{O} &= \mathbf{X} \mathbf{W} + \mathbf{b}, \\ \hat{\mathbf{Y}} & = \mathrm{softmax}(\mathbf{O}). \end{aligned} $$

:eqlabel:`eq_minibatch_softmax_reg`

Esto acelera la operación dominante en una matriz--producto de matriz $\mathbf{X} \mathbf{W}$. Además, ya que cada fila en $\mathbf{X}$ representa un ejemplo de datos, la operación softmax en sí puede ser computada * arrowwise* para cada fila de $\mathbf{O}$, exponente todas las entradas y luego normalizarlas por la suma. Tenga en cuenta, sin embargo, que se debe tener cuidado para evitar exponenciar y tomar logaritmos de grandes números, ya que esto puede causar desbordamiento numérico o subflujo. Marcos de aprendizaje profundo se ocupan de esto automáticamente.

## Función de pérdida
<a id="subsec_softmax-regression-loss-func"></a>

Ahora que tenemos una asignación de características $\mathbf{x}$ a probabilidades $\mathbf{\hat{y}}$, necesitamos una manera de optimizar la precisión de esta asignación. Nos basaremos en la estimación de máxima probabilidad, el mismo método que encontramos al proporcionar una justificación probabilística para la pérdida de error cuadrado media en
[Referencia subsec_normal_distribution_and_squared_loss](https://d2l.ai/chapter_linear-regression/linear-regression.html#subsec-normal-distribution-and-squared-loss).

### Log-verosimilitud
La función softmax nos da un vector $\hat{\mathbf{y}}$, que podemos interpretar como las probabilidades condicionales (estimadas) de cada clase, dada cualquier entrada $\mathbf{x}$, como $\hat{y}_1$ = $P(y=\textrm{cat} \mid \mathbf{x})$. En lo siguiente suponemos que para un conjunto de datos con características $\mathbf{X}$ las etiquetas $\mathbf{Y}$ se representan usando un vector de etiqueta de codificación de un solo calor. Podemos comparar las estimaciones con la realidad comprobando cuán probables son las clases reales según nuestro modelo, dadas las características:

$$
P(\mathbf{Y} \mid \mathbf{X}) = \prod_{i=1}^n P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)}).
$$

Se nos permite utilizar la factorización ya que suponemos que cada etiqueta se extrae independientemente de su respectiva distribución $P(\mathbf{y}\mid\mathbf{x}^{(i)})$. Dado que maximizar el producto de los términos es incómodo, tomamos el logaritmo negativo para obtener el problema equivalente de minimizar el logaritmo negativo:

$$
-\log P(\mathbf{Y} \mid \mathbf{X}) = \sum_{i=1}^n -\log P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})
= \sum_{i=1}^n l(\mathbf{y}^{(i)}, \hat{\mathbf{y}}^{(i)}),
$$

donde para cualquier par de etiquetas $\mathbf{y}$ y predicción de modelos $\hat{\mathbf{y}}$ sobre $q$ clases, la función de pérdida $l$ es

$$ l(\mathbf{y}, \hat{\mathbf{y}}) = - \sum_{j=1}^q y_j \log \hat{y}_j. $$

:eqlabel:`eq_l_cross_entropy`

Por razones que se explican más adelante, la función de pérdida en [Referencia eq_l_cross_entropy](https://d2l.ai/#eq-l-cross-entropy) se llama comúnmente la *pérdida de entropía cruzada*. Puesto que $\mathbf{y}$ es un vector de longitud $q$, la suma sobre todas sus coordenadas $j$ desaparece para todos menos un término. Tenga en cuenta que la pérdida $l(\mathbf{y}, \hat{\mathbf{y}})$ se limita desde abajo por $0$ siempre que $\hat{\mathbf{y}}$ es un vector de probabilidad: ninguna entrada única es mayor que $1$, por lo que su logaritmo negativo no puede ser inferior a $0$; $l(\mathbf{y}, \hat{\mathbf{y}}) = 0$ sólo si predecimos la etiqueta real con *certainty*. Esto nunca puede ocurrir para cualquier ajuste finito de los pesos porque tomar una salida softmax hacia $1$ requiere llevar la entrada $o_i$ correspondiente al infinito (o todas las otras salidas $o_j$ para $j \neq i$ a infinito negativo). Incluso si nuestro modelo podría asignar una probabilidad de salida de $0$, cualquier error hecho al asignar tal alta confianza incurriría en pérdida infinita ($-\log 0 = \infty$).

### Pérdida de softmax y entropía cruzada
<a id="subsec_softmax_and_derivatives"></a>

Dado que la función softmax y la pérdida de entropía cruzada correspondiente son tan comunes, vale la pena entender un poco mejor cómo se calculan. Conectar [Referencia eq_softmax_y_and_o](https://d2l.ai/#eq-softmax-y-and-o) en la definición de la pérdida en [Referencia eq_l_cross_entropy](https://d2l.ai/#eq-l-cross-entropy) y utilizando la definición de la softmax que obtenemos

$$
\begin{aligned}
l(\mathbf{y}, \hat{\mathbf{y}}) &=  - \sum_{j=1}^q y_j \log \frac{\exp(o_j)}{\sum_{k=1}^q \exp(o_k)} \\
&= \sum_{j=1}^q y_j \log \sum_{k=1}^q \exp(o_k) - \sum_{j=1}^q y_j o_j \\
&= \log \sum_{k=1}^q \exp(o_k) - \sum_{j=1}^q y_j o_j.
\end{aligned}
$$

Para entender un poco mejor lo que está pasando, considere la derivada con respecto a cualquier logit $o_j$.

$$
\partial_{o_j} l(\mathbf{y}, \hat{\mathbf{y}}) = \frac{\exp(o_j)}{\sum_{k=1}^q \exp(o_k)} - y_j = \mathrm{softmax}(\mathbf{o})_j - y_j.
$$

En otras palabras, la derivada es la diferencia entre la probabilidad asignada por nuestro modelo, expresada por la operación softmax, y lo que realmente sucedió, expresado por elementos en el vector de la etiqueta de un solo calor. En este sentido, es muy similar a lo que vimos en regresión, donde el gradiente fue la diferencia entre la observación $y$ y la estimación $\hat{y}$. Esto no es una coincidencia. En cualquier modelo familiar exponencial, los gradientes de la semejanza de log son dados precisamente por este término. Este hecho hace que los gradientes de computación sean fáciles en la práctica.

Ahora consideremos el caso donde observamos no sólo un único resultado sino una distribución completa sobre los resultados. Podemos utilizar la misma representación que antes para la etiqueta $\mathbf{y}$. La única diferencia es que en lugar de un vector que contiene sólo entradas binarias, digamos $(0, 0, 1)$, ahora tenemos un vector de probabilidad genérico, digamos $(0.1, 0.2, 0.7)$. La matemática que usamos anteriormente para definir la pérdida $l$ en [Referencia eq_l_cross_entropy](https://d2l.ai/#eq-l-cross-entropy) todavía funciona bien, sólo que la interpretación es ligeramente más general. Es el valor esperado de la pérdida para una distribución sobre las etiquetas. Esta pérdida se llama la *pérdida de entropía cruzada* y es una de las pérdidas más utilizadas para los problemas de clasificación. Podemos desmitificar el nombre introduciendo sólo los fundamentos de la teoría de la información. En pocas palabras, mide el número de bits necesarios para codificar lo que vemos, $\mathbf{y}$, en relación con lo que predecimos que debe suceder, $\hat{\mathbf{y}}$. Proporcionamos una explicación muy básica en el siguiente.
[Cover.Thomas.1999](https://d2l.ai/chapter_references/zreferences.html) o [mackay2003information](https://d2l.ai/chapter_references/zreferences.html).

## Fundamentos de la Teoría de la Información
<a id="subsec_info_theory_basics"></a>

Muchos documentos de aprendizaje profundo utilizan la intuición y los términos de la teoría de la información. Para darles sentido, necesitamos un lenguaje común. Esta es una guía de supervivencia. *Teoría de la información* trata el problema de codificar, decodificar, transmitir y manipular información (también conocida como datos).

### Entropía
La idea central en la teoría de la información es cuantificar la cantidad de información contenida en los datos. Esto pone un límite a nuestra capacidad de comprimir datos. Para una distribución $P$ su *entropía*, $H[P]$, se define como:

$$H[P] = \sum_j - P(j) \log P(j).$$

:eqlabel:`eq_softmax_reg_entropy`

Uno de los teoremas fundamentales de la teoría de la información afirma que para codificar los datos extraídos aleatoriamente de la distribución $P$, necesitamos al menos $H[P]$ "nats" para codificarlo [Shannon.1948](https://d2l.ai/chapter_references/zreferences.html). Si te preguntas qué es un "nat", es el equivalente de bit pero cuando se utiliza un código con base $e$ en lugar de uno con base 2. Así, un nat es $\frac{1}{\log(2)} \approx 1.44$ bit.

### Surprisal
Usted puede estar preguntándose qué tiene que ver la compresión con la predicción. Imagine que tenemos una corriente de datos que queremos comprimir. Si siempre es fácil para nosotros predecir el siguiente token, entonces estos datos son fáciles de comprimir. Tome el ejemplo extremo donde cada token en la corriente siempre toma el mismo valor. Eso es una corriente de datos muy aburrido! Y no sólo es aburrido, pero también es fácil de predecir. Debido a que los tokens son siempre lo mismo, no tenemos que transmitir ninguna información para comunicar el contenido de la corriente. Fácil de predecir, fácil de comprimir.

Sin embargo, si no podemos predecir perfectamente cada evento, entonces a veces podemos sorprendernos. Nuestra sorpresa es mayor cuando se asigna un evento menor probabilidad. Claude Shannon se estableció en $\log \frac{1}{P(j)} = -\log P(j)$ para cuantificar su *surprisal* al observar un evento $j$ habiéndolo asignado una probabilidad (subjetiva) $P(j)$. La entropía definida en [Referencia eq_softmax_reg_entropy](https://d2l.ai/#eq-softmax-reg-entropy) es entonces el *surprisal esperado* cuando se asignan las probabilidades correctas que realmente coinciden con el proceso de generación de datos.

### Entropía cruzada revisitada
Así que si la entropía es el nivel de sorpresa experimentado por alguien que conoce la verdadera probabilidad, entonces usted podría estar preguntándose, ¿qué es la entropía cruzada? La entropía cruzada * de * $P$ * a * $Q$, denotado $H(P, Q)$, es la sobrepresación esperada de un observador con probabilidades subjetivas $Q$ al ver los datos que se generaron realmente de acuerdo con las probabilidades $P$. Esto se da por $H(P, Q) \stackrel{\textrm{def}}{=} \sum_j - P(j) \log Q(j)$. La menor posible entropía cruzada se logra cuando $P=Q$. En este caso, la entropía cruzada de $P$ a $Q$ es $H(P, P)= H(P)$.

En resumen, podemos pensar en el objetivo de clasificación de la entropía cruzada de dos maneras: (i) como maximizar la probabilidad de los datos observados; y (ii) como minimizar nuestra sobrepresal (y por lo tanto el número de bits) requerido para comunicar las etiquetas.

## Resumen y debate
En esta sección, nos encontramos con la primera función de pérdida no trivial, lo que nos permite optimizar sobre *discreto* espacios de salida. La clave en su diseño fue que tomamos un enfoque probabilístico, tratando categorías discretas como casos de sorteos de una distribución de probabilidad. Como efecto secundario, nos encontramos con el softmax, una conveniente función de activación que transforma salidas de una capa normal de red neuronal en distribuciones de probabilidad discretas válidas. Vimos que la derivada de la pérdida de la entropía cruzada cuando se combina con softmax se comporta muy similar a la derivada de error cuadrado; es decir, tomando la diferencia entre el comportamiento esperado y su predicción. Y, mientras que sólo pudimos rascar la superficie misma de ella, nos encontramos con emocionantes conexiones a la física estadística y la teoría de la información.

Mientras que esto es suficiente para conseguir que en su camino, y con suerte suficiente para avivar su apetito, apenas buceamos profundo aquí. Entre otras cosas, nos saltamos sobre consideraciones computacionales. Específicamente, para cualquier capa totalmente conectada con $d$ entradas y $q$ salidas, la parametrización y el costo computacional es $\mathcal{O}(dq)$, que puede ser prohibitivamente alto en la práctica. Afortunadamente, este costo de transformar entradas $d$ en $q$ salidas se puede reducir a través de aproximación y compresión. Por ejemplo, Convnets Fríos Profundos [Yang.Moczulski.Denil.ea.2015](https://d2l.ai/chapter_references/zreferences.html) utiliza una combinación de permutaciones, Fourier transforma, y escala para reducir el costo de cuadrático a log-lineal. Técnicas similares trabajan para aproximaciones de matriz estructural más avanzadas [sindhwani2015structured](https://d2l.ai/chapter_references/zreferences.html). Por último, podemos utilizar descomposicións tipo cuaternión para reducir el costo a $\mathcal{O}(\frac{dq}{n})$, de nuevo si estamos dispuestos a negociar una pequeña cantidad de precisión para calcular y el costo de almacenamiento [Zhang.Tay.Zhang.ea.2021](https://d2l.ai/chapter_references/zreferences.html) basado en un factor de compresión $n$. Este es un área activa de investigación.



### Nota docente de Hespérides

Para logits $z$ y clase correcta $y$, la entropía cruzada es $\log\sum_j e^{z_j}-z_y$. Restar el máximo antes de exponenciar evita desbordamientos sin cambiar las probabilidades. `CrossEntropyLoss` recibe logits: aplicar softmax antes cambia la entrada que espera. MSE corresponde a otra hipótesis sobre los datos; la elección de pérdida expresa cómo modelamos el error, no solo qué curva nos resulta cómoda.

Vínculo con los apuntes: sesión 2, «Softmax, entropía cruzada y pérdida probabilística».


**Errata del original:** en el ejercicio de log-partición, la igualdad de invariancia planteada debe revisarse. Comprueba su validez antes de intentar demostrarla.

## Ejercicios
1. Podemos explorar la conexión entre familias exponenciales y softmax en una mayor profundidad.
    1. Calcular la segunda derivada de la pérdida de entropía cruzada $l(\mathbf{y},\hat{\mathbf{y}})$ para softmax.
    1. Calcular la varianza de la distribución dada por $\mathrm{softmax}(\mathbf{o})$ y mostrar que coincide con la segunda derivada calculada anteriormente.
1. Supongamos que tenemos tres clases que ocurren con igual probabilidad, es decir, el vector de probabilidad es $(\frac{1}{3}, \frac{1}{3}, \frac{1}{3})$.
    1. ¿Cuál es el problema si tratamos de diseñar un código binario para él?
    1. ¿Puede diseñar un código mejor? Consejo: ¿qué pasa si tratamos de codificar dos observaciones independientes? ¿Qué pasa si codificamos conjuntamente las observaciones $n$?
1. Al codificar señales transmitidas a través de un cable físico, los ingenieros no siempre usan códigos binarios. Por ejemplo, [PAM-3](https://en.wikipedia.org/wiki/Ternary_signal) utiliza tres niveles de señal $\{-1, 0, 1\}$ en lugar de dos niveles $\{0, 1\}$. ¿Cuántas unidades ternarias necesita para transmitir un entero en el rango $\{0, \ldots, 7\}$? ¿Por qué podría ser esta una mejor idea en términos de electrónica?
1. El [Bradley--Terry model](https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model) utiliza
un modelo logístico para capturar preferencias. Para que un usuario pueda elegir entre manzanas y naranjas se asumen las puntuaciones $o_{\textrm{apple}}$ y $o_{\textrm{orange}}$. Nuestros requisitos son que las puntuaciones más grandes deben conducir a una mayor probabilidad en la elección del elemento asociado y que el elemento con la puntuación más grande es el más probable a ser elegido [Bradley.Terry.1952](https://d2l.ai/chapter_references/zreferences.html).
    1. Demostrar que softmax satisface este requisito.
    1. ¿Qué pasa si quieres permitir una opción predeterminada de elegir ni manzanas ni naranjas? Sugerencia: ahora el usuario tiene tres opciones.
1. Softmax obtiene su nombre de la siguiente asignación: $\textrm{RealSoftMax}(a, b) = \log (\exp(a) + \exp(b))$.
    1. Demostrar que $\textrm{RealSoftMax}(a, b) > \mathrm{max}(a, b)$.
    1. ¿Qué tan pequeño se puede hacer la diferencia entre ambas funciones? Consejo: sin pérdida de generalidad se puede establecer $b = 0$ y $a \geq b$.
    1. Demostrar que esto es válido para $\lambda^{-1} \textrm{RealSoftMax}(\lambda a, \lambda b)$, siempre que $\lambda > 0$.
    1. Mostrar que para $\lambda \to \infty$ tenemos $\lambda^{-1} \textrm{RealSoftMax}(\lambda a, \lambda b) \to \mathrm{max}(a, b)$.
    1. Construir una función softmin análoga.
    1. Extienda esto a más de dos números.
1. La función $g(\mathbf{x}) \stackrel{\textrm{def}}{=} \log \sum_i \exp x_i$ a veces también se conoce como la [log-partition function](https://en.wikipedia.org/wiki/Partition_function_(mathematics)).
    1. Probar que la función es convexa. Consejo: para hacerlo, utilice el hecho de que la primera derivada equivale a las probabilidades de la función softmax y mostrar que la segunda derivada es la varianza.
    1. Mostrar que $g$ es invariante de traducción, es decir, $g(\mathbf{x} + b) = g(\mathbf{x})$.
    1. ¿Qué pasa si algunas de las coordenadas $x_i$ son muy grandes? ¿Qué pasa si todas son muy pequeñas?
    1. Demostrar que si elegimos $b = \mathrm{max}_i x_i$ terminamos con una implementación numéricamente estable.
1. Supongamos que tenemos alguna distribución de probabilidad $P$. Supongamos que escogemos otra distribución $Q$ con $Q(i) \propto P(i)^\alpha$ para $\alpha > 0$.
    1. ¿Qué elección de $\alpha$ corresponde a duplicar la temperatura? ¿Qué elección corresponde a reducirla a la mitad?
    1. ¿Qué pasa si dejamos que la temperatura se aproxime a $0$?
    1. ¿Qué pasa si dejamos que la temperatura se aproxime a $\infty$?

[Debate del original](https://discuss.d2l.ai/t/46)
